<a href="https://colab.research.google.com/github/carlolzz/llm-from-scratch/blob/main/chapter_3_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Simplified Attention

Example taken from the book

In [ ]:
import torch
inputs = torch.tensor(
 [[0.43, 0.15, 0.89],   # Your    (x^1)
 [0.55, 0.87, 0.66],    # journey (x^2)
 [0.57, 0.85, 0.64],    # starts  (x^3)
 [0.22, 0.58, 0.33],    # with    (x^4)
 [0.77, 0.25, 0.10],    # one     (x^5)
 [0.05, 0.80, 0.55]]    # step    (x^6)
)

In [ ]:
# 6x3 matrix
inputs.shape

torch.Size([6, 3])

As an example we calculate the *attention scores* $ω$ between the second token and all other tokens.

In [ ]:
idx:int = 1
# [0.55, 0.87, 0.66]
query: list[float] = inputs[idx]
attention_scores_2: list[float] = torch.empty(inputs.shape[0])
for idx, x_i in enumerate(inputs):
    # Attention scores for the second input token, "journey"
    attention_scores_2[idx] = torch.dot(query, x_i)

print(f"Attention scores: {attention_scores_2}")
# Contains all of the dot products between the second word vector (its tokens) and all the other word vectors

Attention scores: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


The dot product is a mathematical tool that combines two vector and measures their similarity, by seeing how close they align. <br>The formula is defined as: $[ \vec{p} \cdot \vec{q} ]$, or also $[ \vec{p} \cdot \vec{q}] =\left | \vec{p} \right |\left | \vec{q} \right |\cos\theta$<br>
We then normalize these attention scores, so that they sum up to 1.

In [ ]:
# Normalizing the attention scores, so that they sum up to 1
# Dividing each element in the vector by the sum of the elements of the vector
attention_weights_2_tmp = attention_scores_2 / attention_scores_2.sum()
print(f"Attention weights: {attention_weights_2_tmp}")
print(f"Sum: {attention_weights_2_tmp.sum():1f}")

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: 1.000000


####Softmax

In this case it's much better to use the softmax function

$\sigma(z_i) = \frac{e^{z_{i}}}{\sum_{j=1}^K e^{z_{j}}} \ \ \ for\ i=1,2,\dots,K$

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

# dim=0: sums values over columns (if it were a matrix, in this case of a vector dim=1 and dim=0 would be the same)
# in our case a row represents a single vector of one tokens of the sequence

In [ ]:
att_weights = softmax_naive(attention_scores_2)
print(f"Attention weights: {att_weights}")
print(f"Sum: {att_weights.sum():1f}")

# This is the softmax score of similarity of "journey" with all other words

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.000000


Proper softmax implementation

In [ ]:
attn_weights_2 = torch.softmax(attention_scores_2, dim=0)
print(f"Attention weights {attn_weights_2}")
print(f"Sum: {attn_weights_2.sum()}")

Attention weights tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: 1.0


What we are doing: we are calculating the context vector $z^{(2)}$ by multiplying the input tokens $x^{(i)}$'s embeddings with the second input token, obtaining the attention score of all other tokens with respect to the second token. We then normalize these scores to obtain attention weights, normalizing them via softmax so that a single row sums up to 1 (the sum of all other token attention values to the second token sum up to 1). The attention weights represents how much each token "attends to" the second token. Finally, we multiply the attention weights with the corresponding initial input embeddings, sum them up, and obtain the final context vector $z^{(2)}$. It's the second context vector because the attention weights were computed w.r.t the second input vector.

In [ ]:
query = inputs[idx]
context_vec_2 = torch.zeros(query.shape)
for idx, x_i in enumerate(inputs):
    context_vec_2 = context_vec_2 + attn_weights_2[idx] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


Matrix of attention weights, for all input tokens

In [ ]:
attention_matrix = torch.empty((inputs.shape[0], inputs.shape[0])) # 6x6 tensor
attention_matrix

tensor([[3.2687e+21, 4.1763e-08, 1.3304e-05, 2.3051e-12, 1.8788e+31, 7.9303e+34],
        [6.1949e-04, 1.8590e+34, 7.7767e+31, 7.1536e+22, 3.3803e-18, 2.0552e+32],
        [1.8755e+28, 3.1093e-18, 2.0552e+32, 1.8755e+28, 3.1093e-18, 1.2845e+31],
        [1.8395e+25, 6.1963e-04, 1.7560e-04, 8.4265e-07, 6.4519e-07, 2.6078e-09],
        [4.2130e+21, 8.1262e+20, 8.1265e+20, 1.0733e-08, 2.4312e-18, 1.1963e+22],
        [3.1097e-18, 9.4370e-09, 1.0357e-11, 1.0082e-08, 6.6060e-07, 6.7973e-04]])

###Attention scores

In [ ]:
for idx, x_i in enumerate(inputs):
    for jdx, x_j in enumerate(inputs):
        attention_matrix[idx, jdx] = torch.dot(x_i, x_j)

print(f"Attention scores: {attention_matrix}")

# Attention matrix [0, 0]: attention score of the 1st input token to the first input token
# Attention matrix [0, 1]: attention score of the 1st input token to the second input token
# Attention matrix [3, 4]: attention score of the 3rd input token to the 4th input token
assert attention_matrix[3, 4] == torch.dot(inputs[3], inputs[4])

Attention scores: tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


More easily computed as

In [ ]:
# Matrix multiplication between the input matrix and its transpose
# Row column multiplication and then sum up the results
attention_weights = torch.softmax((inputs @ inputs.T), dim=-1)
print(attention_weights)

# dim specifies the dimension of the input tensor along which the function will be computed
# dim=-1, the softmax function will be applied to the last dimension of the attention_scores tensor
# We will therefore normalize across the columns (of each row at a time) so that the values in each row sum up to 1
# Earlier we did dim=0, because it was the only dimension present

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


Matrix row-column multiplication $ c_{ij}= a_{i1} b_{1j} + a_{i2} b_{2j} +\cdots+ a_{in} + b_{nj} = \sum_{k=1}^n a_{ik}b_{kj} $

Applying the softmax to the rows (across the columns of each row at a time) ensures that for each word, the total attention it distributes across the entire sequence sums up to 1.

`dim=0` (first dimension, typically rows): The operation is applied along this dimension. For a sum or softmax, this means it operates on values within each column. The dimension 0 is then 'squashed' or reduced in the output.<br>

`dim=1` (second dimension, typically columns): The operation is applied along this dimension. For a sum or softmax, this means it operates on values within each row.
The dimension 1 is then 'squashed' or reduced in the output.
`dim`: The dimension you specify is the one along which the function is computed, and typically gets reduced or removed in the output shape.

In [ ]:
# [0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452]
# This is the first row
# attention_scores[row][column]

attention_weights.shape
# 6x6 matrix, 6 rows (dimension 0) and 6 columns (dimension 1)

print(f"First row:    {attention_weights[0, :]}")  # → Returns the first row (the first horizontal line).
print(f"First column: {attention_weights[:, 0]}")  # → Returns the first column (the first vertical slice).

First row:    tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452])
First column: tensor([0.2098, 0.1385, 0.1390, 0.1435, 0.1526, 0.1385])


Verifying that it is indeed correct

In [ ]:
# Result for index 0
row_i_sum = sum(attention_weights[0][idx] for idx in range(attention_weights.shape[1]))
print(row_i_sum)

# For all
print(attention_weights.sum(dim=-1))

tensor(1.0000)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


###Final Context vectors

Final output tensor. Each row is a 3 dimensional context vector.
We're multipling the matrix we've just created, $W_{ij}$, with the initial input matrix. We obtain a new matrix with shape $6\times3$, with each row being the new context vector.

In [ ]:
all_context_vecs = attention_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


##Self Attention

*scaled dot-product attention*

$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

In [ ]:
x_2 = inputs[1]         # Second input element, "journey"
d_in = inputs.shape[1]  # Input embedding size, 3 in this case
d_out = 2               # Output embedding size, set at 2 for demonstration purposes

In [ ]:
torch.manual_seed(123)
# Parameter makes it trainable
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# Set as false for demonstration purposes only

Calculate query, key, and value vectors.
We calculate the query vector $q$ only for the second input token, in this case.

In [ ]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


From $6$ inputs tokens of a $3d$ dimensional space down to a $2$ dimensional space.<br>
$(6×3) (3×2) → 6×2$, which is the final matrix shape.

In [ ]:
keys = inputs @ W_key
values = inputs @ W_value

print(f"keys shape:   {keys.shape}")
print(f"values shape: {values.shape}")

keys shape:   torch.Size([6, 2])
values shape: torch.Size([6, 2])


Attention score $ω_{22}$

In [ ]:
keys_2 = keys[1]
attn_scores_22 = query_2.dot(keys_2)
print(attn_scores_22)

tensor(1.8524)


In [ ]:
# Generalized for the complete second input element
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


We compute the attention weights by scaling the attention scores and using the softmax function

In [ ]:
# d_k = d_out
d_k = keys.shape[-1] # 2
attn_weights_2 = torch.softmax((attn_scores_2 / (d_k ** 0.5)), dim=-1)
print(attn_weights_2)

# The normalization is done to improve the training performance

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


###Explanation

In this example, we are calculating the context vector $z^{(2)}$ of the second input token $x^{(2)}$ which represents an enriched representation of the token with information from all other tokens in the same sequence. We calculate its respective query vector $q^{(2)}$, by multiplying $x^{(2)}$ with the projection matrix $W_q$. We then multiply each token $x^{(i)}$ by the weight projection matrices $W_k$ and $W_v$ to obtain the corresponding key and value vectors, $k^{(i)}$ and $v^{(i)}$.

We multiply the key vectors with the query vector $q^{(2)}$, to obtain the corresponding attention scores $ω^{(21)}, ω^{(22)}$... and so on. These attention scores are first scaled, then normalized thanks to a softmax function, to obtain the final attention weights $α^{(12)}, α^{(22)}, ...$, which indicate how much the second query token (in this case) *attends to* each other token. We multiply each attention weight with their respective value vectors $v$ and we compute a weighted sum of the resulting weighted value vectors to obtain the final context vector $z^{(2)}$.

In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


The *query* represents what the model is currently looking for. It represents the specific word or concept the model is focusing on right now.<br><br>
The *key*, on the other hand, serves like a descriptive label or index. Every word in the sequence has its own assigned key that encapsulates what kind of information it holds. The model compares the current query against all available keys to figure out which words are most relevant to each other.<br><br>
The *value* is the actual underlying content, much like the value in a dictionary's key-value pair. Once the model determines which keys best match its query, it retrieves those corresponding values.<br><br>
Ultimately, we want the model to learn exactly which aspects of a word's meaning matter for the task at hand. Because of this, the final value vectors act as filtered versions of the original word embeddings, passing forward only the specific information that is relevant to the current context.

###Class

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values= x @ self.W_value
        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(
            attention_scores / ((self.W_key.shape[-1] ** 0.5)), dim=-1
        )
        context_vector = attention_weights @ values
        return context_vector

Trying it out

In [ ]:
torch.manual_seed(123)
self_att_v1 = SelfAttention_v1(d_in, d_out)
print(self_att_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # nn.Linear performs x @ W^T + bias.
        # This is why the weight matrix shape inside nn.Linear is (d_out, d_in)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(
            attention_scores / ((keys.shape[-1] ** 0.5)), dim=-1
        )
        context_vector = attention_weights @ values
        return context_vector

In [ ]:
torch.manual_seed(789)
self_att_v2 = SelfAttention_v2(d_in, d_out)
print(self_att_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


##Causal attention

In [ ]:
self_att_tmp = SelfAttention_v2(d_in, d_out)

In [ ]:
queries = self_att_tmp.W_query(inputs)
keys = self_att_tmp.W_key(inputs)
values = self_att_tmp.W_value(inputs)

attn_scores_tmp = queries @ keys.T
attn_weights_tmp = torch.softmax(
    attn_scores_tmp / (keys.shape[-1] ** 0.5), dim=-1
)

In [ ]:
attn_weights_tmp

tensor([[0.1766, 0.1701, 0.1699, 0.1597, 0.1618, 0.1620],
        [0.1772, 0.1720, 0.1717, 0.1580, 0.1596, 0.1615],
        [0.1769, 0.1719, 0.1716, 0.1582, 0.1597, 0.1616],
        [0.1725, 0.1696, 0.1695, 0.1618, 0.1627, 0.1638],
        [0.1687, 0.1694, 0.1692, 0.1637, 0.1634, 0.1656],
        [0.1758, 0.1704, 0.1702, 0.1598, 0.1615, 0.1623]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
context_length = attn_scores_tmp.shape[0] # 6
# Creating a mask, by using a triangular matrix of ones and zeros
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
masked_attn_weights = attn_weights_tmp * mask_simple
masked_attn_weights

tensor([[0.1766, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1772, 0.1720, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1769, 0.1719, 0.1716, 0.0000, 0.0000, 0.0000],
        [0.1725, 0.1696, 0.1695, 0.1618, 0.0000, 0.0000],
        [0.1687, 0.1694, 0.1692, 0.1637, 0.1634, 0.0000],
        [0.1758, 0.1704, 0.1702, 0.1598, 0.1615, 0.1623]],
       grad_fn=<MulBackward0>)

We still want the values in each row to sum up to 1, for optimization purposes

In [ ]:
# We just perform a sum along the columns, dim=-1, and then divide by the sum
row_sums = masked_attn_weights.sum(dim=-1, keepdim=True)
masked_simple_norms = masked_attn_weights / row_sums
masked_simple_norms

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5075, 0.4925, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3399, 0.3303, 0.3298, 0.0000, 0.0000, 0.0000],
        [0.2562, 0.2519, 0.2517, 0.2402, 0.0000, 0.0000],
        [0.2021, 0.2030, 0.2028, 0.1962, 0.1959, 0.0000],
        [0.1758, 0.1704, 0.1702, 0.1598, 0.1615, 0.1623]],
       grad_fn=<DivBackward0>)

A trick is to set the values on the diagonal to $-∞$, instead of $0$. This way when we apply softmax, it works correctly.

In [ ]:
# We use attention scores
# Creating a mask with 1s above the diagonal, then setting them to -inf
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
# Sets the top right diagonal entries to minus infinite
# Treat the mask we have just created as a mask of True and False, 1 is True
masked = attn_scores_tmp.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2118,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.2676, 0.2249,   -inf,   -inf,   -inf,   -inf],
        [0.2622, 0.2215, 0.2193,   -inf,   -inf,   -inf],
        [0.1496, 0.1257, 0.1244, 0.0587,   -inf,   -inf],
        [0.0926, 0.0984, 0.0972, 0.0506, 0.0479,   -inf],
        [0.2108, 0.1664, 0.1649, 0.0754, 0.0907, 0.0973]],
       grad_fn=<MaskedFillBackward0>)

In [ ]:
# Attention calculations
# softmax((Q * K^T)) / (sqrt(d)) * V
attn_weights_tmp = torch.softmax((masked / (d_k ** 0.5)), dim=-1)
print(attn_weights_tmp)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5075, 0.4925, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3399, 0.3303, 0.3298, 0.0000, 0.0000, 0.0000],
        [0.2562, 0.2519, 0.2517, 0.2402, 0.0000, 0.0000],
        [0.2021, 0.2030, 0.2028, 0.1962, 0.1959, 0.0000],
        [0.1758, 0.1704, 0.1702, 0.1598, 0.1615, 0.1623]],
       grad_fn=<SoftmaxBackward0>)


###Dropout

We can add a dropout mask so the model learns to rely less on position

In [ ]:
# torch.nn.Dropout(dropout_percentage)
torch.manual_seed(123)
dropout_layer = torch.nn.Dropout(0.5) # dropout rate of 50%

In [ ]:
example = torch.ones(6, 6)
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [ ]:
dropout_layer(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

Some values are now larger, why? To maintain the same sum in each row, it's rescaling the values.<br>
$\dfrac{1}{1 - dropout\_rate}$


Dropout applied to the attention weights

In [ ]:
torch.manual_seed(123)
print(dropout_layer(attn_weights_tmp))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6799, 0.6606, 0.6595, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5038, 0.5033, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4060, 0.0000, 0.3924, 0.0000, 0.0000],
        [0.0000, 0.3408, 0.3404, 0.3196, 0.3230, 0.0000]],
       grad_fn=<MulBackward0>)


###Updated Attention Class

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
batch
# batch shape: [2, 6, 3]
# 2 6x3 matrices stacked on top of each other

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [ ]:
# Batch size, number of tokens (in each row), dimension in input
batch.shape

torch.Size([2, 6, 3])

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        # nn.Linear performs x @ W^T + bias.
        # This is why the weight matrix shape inside nn.Linear is (d_out, d_in)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # Causal mask, register as a buffer
        # In pytorch, we can move a model to a device (cuda gpu)
        # With a buffer we can also move the mask to the gpu
        self.register_buffer(
            "mask", torch.triu((torch.ones(context_length, context_length)), diagonal=1)
        )

    def forward(self, x):
        # Batch, number of tokens, dimension in input
        # Batch of 2, 6 tokens in each row, dimension in input of 3
        # -- each input token is represented by a 3 dimensional embedding
        _, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We transpose dimension 1 and 2,
        # We keep don't alter the batch dimension, keeping it at index 0 (first position)
        attention_scores = queries @ keys.transpose(1, 2)
        # masked.fill_ = inplace operation for optimization
        attention_scores.masked_fill_(
            # In this example, num_tokens = 6
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attention_weights = torch.softmax(
            attention_scores / ((keys.shape[-1] ** 0.5)), dim=-1
        )
        attention_weights = self.dropout(attention_weights)

        context_vector = attention_weights @ values
        return context_vector

In [ ]:
context_lenght = batch.shape[1]
dropout_rate = 0.0

In [ ]:
torch.manual_seed(123)
causal_att = CausalAttention(d_in, d_out, context_length, dropout_rate)

In [ ]:
context_vectors = causal_att(batch)
context_vectors

print(f"context vectors shape: {context_vectors.shape}")

context vectors shape: torch.Size([2, 6, 2])


###Multi-head Attention

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias= False):
        super().__init__()
        # Stacking causal attention mechanism
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = batch.shape[-1], 2
dropout_rate = 0.0
num_heads = 2
multi_head_att = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout_rate, num_heads)

In [ ]:
context_vecs_temp = multi_head_att(batch)
print(context_vecs_temp)
print(f"\nShape of the context vectors after applying multi head attention: {context_vecs_temp.shape}")

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

Shape of the context vectors after applying multi head attention: torch.Size([2, 6, 4])


###Weight Splits

####Explanation

Instead of performing multiple matrix multiplication, we stack multiple matrices like 2 $W_q$, multiply it by the $2$ stacked $W_k$ matrices and then we split them up.<br>

All the transposes and view operations are complex but they serve an important role. In `pytorch`, the first two dimensions of a 4D tensor are considered the batch *dimension*, and when we do use the `@` operation it only multiplies the last two dimensions.

**1st Step**
```
keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
```
Initial current shape: `(batch, tokens, heads, head_dim)`
<br><br>

**2nd Step**
```
keys = keys.transpose(1, 2)
# same for queries and values
```
Here we swap the 1st and 2nd dimension (tokens and heads), indexes 1 and 2.

New shape: `(batch, heads, tokens, head_dim)`
<br><br>

**3rd step**
```
attn_scores = queries @ keys.transpose(2, 3)
```
Afterwards, since we need to multiply the queries by the keys, we need the inner dimensions to match, but:
*   queries shape: `(batch, heads, tokens, head_dim)`
*   keys shape: `(batch, heads, tokens, head_dim)`
<br>

Therefore we need keys to be `(batch, heads, head_dim, tokens)`

What is being multiplied: `(tokens, head_dim) @ (head_dim, tokens) = (tokens, tokens)`
Resulting attn_scores shape: `(batch, heads, tokens, tokens)`<br>
<br><br>
**4th step**<br>
`context_vec = (attn_weights @ values).transpose(1, 2)`<br>
We need to stitch everything back together. We transpose dimensions 1 and 2 again to move the tokens back to their original place.

New shape: `(batch, tokens, heads, head_dim)`<br>
<br><br>
**5th step**<br>
`context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)`
By using `view` we turn back to a single `d_out` dimension, (since `heads * head_dim = d_out`). The `.contiguous()` is just a PyTorch quirk: transposing messes with how memory is stored in RAM, and `.contiguous()` safely realigns it so `.view()` doesn't crash.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias= False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        # Stacking causal attention mechanism
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
            diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We split the matrix and unroll the last dimension
        # From (b, num_tokens, d_out) to (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(
            b, num_tokens, self.num_heads, self.head_dim
        )

        # From (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)
        # 2nd and 3rd element get flipped (index 1 and 2)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Like before, calculating the attention scores, masking, calculate weights
        # Computing the dot product for each head
        attn_scores = queries @ keys.transpose(2, 3)
        # Mask truncated to the number of tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Using mask to fill out attention scores
        # masked_fill_ operates in-place
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        # Tensor shape = (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(
        b, num_tokens, self.d_out
        )
        context_vec = self.out_proj(context_vec)

        return context_vec

In [ ]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2

In [ ]:
multi_head_att = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = multi_head_att(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
